In [1]:
%pip install ortools

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from ortools.linear_solver import pywraplp
from optimizer.construct_data_objects import SimulationParameters
from optimizer.run_optimizer import construct_supply_chain_data
from optimizer.math_model_declaration import create_math_model
from optimizer.math_model_constraints import minimize_cost_objective
from optimizer.run_optimizer import optimize
import utils.log as log


logger = log.get_logger("SCIPSolver")

Overview- Model is looking at a classic MFG, Distribution, Store Supply Chain optimization.

Each Manufacturing site has a total capacity, and different cost for shipping to each distribution site.

Each Distribution site receives products from a MFG Site, then fulfilled a Store's replenishment demand, with an associated shipping cost. When a distribution site is opened it must remain opened for x number of days

Each distribution has a fixed recurring cost as well as capacity.

Each store has demand that is must fulfill from the MFG site through the distribution to the final store.

Objective is to minimize the total cost of shipping, plus the cost of maintaining a distribution site.

Model:
Demand is randomly generated for each customer for a 10 day period

Global- looks at the total demand minimizes the cost across the entire horizon
Daily - this model is naive and must make decision on a daily, but it inherits decisions from the previous day. Ex. if a distribution site was opened the day before, it must remain open during this time.

RL - we simulated out the model, and try to do daily decision based on our RF model.

Goal is to compare Global, vs daily vs RL and see how well they converge.


In [50]:
num_days=10
num_simulations=10 
decision_rolling_period=3
num_distribution_sites=5

distribution_opening_costs = [350, 320, 375, 400, 550]
# distribution_opening_costs = [1000, 100000, 100000, 100000, 100000]
mfg_site_capacity = [600000, 600000]

mean_demand = [20, 30, 25, 40, 35, 28, 32, 50, 26, 38, 34, 27]
# mean_demand = [100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100]

std_dev_demand = [20, 18, 15, 20, 20, 5, 5, 12.4, 12.6, 13.8, 13.4, 12.7]
# std_dev_demand = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] # If comparing against daily solves or trying to have stable runs, you need to set the std to 0

# Transportation costs
transport_cost_m_to_d = [
    [3.5, 2.5, 4.5, 2.5, 3.0],  # Manufacturing site 1
    [2.5, 4.5, 5.5, 6.5, 8.5]  # Manufacturing site 2
]
transport_cost_d_to_c = [
    [1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2],  # Distribution site 1
    [2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2],  # Distribution site 2
    [2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2],  # Distribution site 3
    [2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2],  # Distribution site 4
    [2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1]   # Distribution site 5
]

# transport_cost_m_to_d = [
#     [3, 3, 3, 3, 3],  # Manufacturing site 1
#     [3, 3, 3, 3, 3]  # Manufacturing site 2
# ]
# transport_cost_d_to_c = [
#     [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],  # Distribution site 1
#     [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],  # Distribution site 2
#     [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],  # Distribution site 3
#     [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],  # Distribution site 4
#     [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]   # Distribution site 5
# ]

In [51]:
%%capture
### Run global solve ###

# Construct supply chain data, params
sim_params = SimulationParameters(num_days, num_simulations, decision_rolling_period)
supply_chain_data = construct_supply_chain_data(mean_demand, mfg_site_capacity, std_dev_demand, distribution_opening_costs, transport_cost_m_to_d, transport_cost_d_to_c)

# Consruct and run optimizer
opening_distribution_costs, total_transport_cost, open_distribution_decisions = optimize(supply_chain_data, sim_params, num_distribution_sites)


2024-04-21 15:09:37 [INFO] (SCIP Solver): Began adding variable distribution_cost_incurred to model
2024-04-21 15:09:37 [DEBUG] (SCIP Solver): Added variable distribution_cost_incurred to model
2024-04-21 15:09:37 [DEBUG] (SCIP Solver): Variable distribution_cost_incurred has 50 variables added.
2024-04-21 15:09:37 [INFO] (SCIP Solver): Began adding variable distribution_on to model
2024-04-21 15:09:37 [DEBUG] (SCIP Solver): Added variable distribution_on to model
2024-04-21 15:09:37 [DEBUG] (SCIP Solver): Variable distribution_on has 50 variables added.
2024-04-21 15:09:37 [INFO] (SCIP Solver): Began adding variable transport_m_to_d to model
2024-04-21 15:09:37 [INFO] (SCIP Solver): Added variable transport_m_to_d to model
2024-04-21 15:09:37 [INFO] (SCIP Solver): Variable transport_m_to_d has 100 variables added.
2024-04-21 15:09:37 [INFO] (SCIP Solver): Began adding variable transport_d_to_c to model
2024-04-21 15:09:37 [INFO] (SCIP Solver): Added variable transport_d_to_c to model


presolving:
(round 1, fast)       20 del vars, 33 del conss, 0 add conss, 1400 chg bounds, 0 chg sides, 0 chg coeffs, 0 upgd conss, 0 impls, 90 clqs
(round 2, fast)       20 del vars, 48 del conss, 0 add conss, 1400 chg bounds, 0 chg sides, 685 chg coeffs, 0 upgd conss, 0 impls, 90 clqs
(round 3, exhaustive) 20 del vars, 53 del conss, 0 add conss, 1400 chg bounds, 5 chg sides, 685 chg coeffs, 0 upgd conss, 0 impls, 90 clqs
(round 4, exhaustive) 20 del vars, 53 del conss, 0 add conss, 1400 chg bounds, 5 chg sides, 685 chg coeffs, 770 upgd conss, 0 impls, 90 clqs
(round 5, exhaustive) 20 del vars, 93 del conss, 0 add conss, 1400 chg bounds, 5 chg sides, 685 chg coeffs, 770 upgd conss, 685 impls, 90 clqs
(round 6, medium)     25 del vars, 93 del conss, 0 add conss, 1400 chg bounds, 10 chg sides, 690 chg coeffs, 770 upgd conss, 685 impls, 90 clqs
   (0.0s) probing cycle finished: starting next cycle
   Deactivated symmetry handling methods, since SCIP was built without symmetry detector (S

2024-04-21 15:09:37 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (1, 7, 0) has value 0.0
2024-04-21 15:09:37 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (1, 7, 1) has value 0.0
2024-04-21 15:09:37 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (1, 7, 2) has value 0.0
2024-04-21 15:09:37 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (1, 7, 3) has value 0.0
2024-04-21 15:09:37 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (1, 7, 4) has value 0.0
2024-04-21 15:09:37 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (1, 7, 5) has value 0.0
2024-04-21 15:09:37 [INFO] (SCIP Solver): Variable t

In [52]:
num_manufacturing_sites = 2
num_distribution_sites = 5
num_customers = 12
num_days = 10
num_simulations = 1
decision_rolling_period = 3

distribution_opening_costs = [350, 320, 375, 400, 550]
mfg_site_capacity = [600000, 600000]

mean_demand = [20, 30, 25, 40, 35, 28, 32, 50, 26, 38, 34, 27]
std_dev_demand = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

# # Transportation costs
transport_cost_m_to_d = [
    [3.5, 2.5, 4.5, 2.5, 3.0],
    [2.5, 4.5, 5.5, 6.5, 8.5]
]
transport_cost_d_to_c = [
    [1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2],  # Distribution site 1
    [2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2],  # Distribution site 2
    [2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2],  # Distribution site 3
    [2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2],  # Distribution site 4
    [2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1]   # Distribution site 5
]


In [53]:
average_dc_open = [0 for _ in range(num_distribution_sites)]
average_difference = []
all_days_demand = []
        
for simulation in range(num_simulations):

    # Reset tracking variables for each simulation
    open_dc_states = [[0 for _ in range(num_days)] for _ in range(num_distribution_sites)]
    open_dc_state = [0 for _ in range(num_distribution_sites)]
    days_dc_open = [0 for _ in range(num_distribution_sites)]

    daily_demand_data = []
    total_daily_cost = 0

    for day in range(num_days):
        # Simulate demand for the day for each customer
        simulated_demand = [max(0, np.random.normal(mean_demand[c], std_dev_demand[c])) for c in range(num_customers)]
        daily_demand_data.append(simulated_demand)
        
        # Run daily simulation
        adj_opening_cost = [0 if state == 1 else cost for state, cost in zip(open_dc_state, distribution_opening_costs)]
        
        # Construct supply chain data, params
        sim_params = SimulationParameters(1, 1, decision_rolling_period)

        # Construct supply chain data
        supply_chain_data = construct_supply_chain_data(simulated_demand, mfg_site_capacity, std_dev_demand, adj_opening_cost, transport_cost_m_to_d, transport_cost_d_to_c)

        # Consruct and run optimizer
        daily_open_distribution_costs, daily_cost, open_distribution_decisions = optimize(supply_chain_data, sim_params, num_distribution_sites)
        
        total_daily_cost += daily_cost

        for d in range(num_distribution_sites):
            open_dc_states[d][day] = open_distribution_decisions[d]
            open_dc_state[d] = open_distribution_decisions[d]

        # print(open_dc_state)
        for distribution in range(num_distribution_sites):
            average_dc_open[distribution] += open_dc_state[distribution]

            if open_dc_state[distribution] == 1:
                days_dc_open[distribution] += 1
            else:
                days_dc_open[distribution] = 0

            if days_dc_open[distribution] >= decision_rolling_period:
                open_dc_state[distribution] = 0
                days_dc_open[distribution] = 0
    
    # Global simulation using the first simulation's demand data
    # Construct supply chain data, params
    sim_params = SimulationParameters(num_days, num_simulations, decision_rolling_period)
    supply_chain_data = construct_supply_chain_data(mean_demand, mfg_site_capacity, std_dev_demand, distribution_opening_costs, transport_cost_m_to_d, transport_cost_d_to_c)

    # Consruct and run optimizer
    opening_distribution_costs, global_cost, global_dc_open = optimize(supply_chain_data, sim_params, num_distribution_sites)
    global_open_distribution_costs, global_cost, global_dc_open = optimize(supply_chain_data, sim_params, distribution_opening_costs)
    difference = total_daily_cost - global_cost
    print(f'Look Back Solve Total:{round(global_cost,2)}, Daily Solve Total:{round(total_daily_cost,2)}, Difference: {round(difference,2)}')
    average_difference.append(difference)

# Calculate average decisions for daily simulations
average_dc_open = [count / (num_simulations * num_days) for count in average_dc_open]

print("Final DC States for Each Decision Point:")
for d in range(num_distribution_sites):
    print(f"Daily Decision Point Dist {d}: {open_dc_states[d]}")
    print(f"Global Decision Point Dist {d}: {global_dc_open[d]}")

print(f"Avg dc open on daily simulation: {average_dc_open}")

2024-04-21 15:09:52 [INFO] (SCIP Solver): Began adding variable distribution_cost_incurred to model
2024-04-21 15:09:52 [DEBUG] (SCIP Solver): Added variable distribution_cost_incurred to model
2024-04-21 15:09:52 [DEBUG] (SCIP Solver): Variable distribution_cost_incurred has 5 variables added.
2024-04-21 15:09:52 [INFO] (SCIP Solver): Began adding variable distribution_on to model
2024-04-21 15:09:52 [DEBUG] (SCIP Solver): Added variable distribution_on to model
2024-04-21 15:09:52 [DEBUG] (SCIP Solver): Variable distribution_on has 5 variables added.
2024-04-21 15:09:52 [INFO] (SCIP Solver): Began adding variable transport_m_to_d to model
2024-04-21 15:09:52 [INFO] (SCIP Solver): Added variable transport_m_to_d to model
2024-04-21 15:09:52 [INFO] (SCIP Solver): Variable transport_m_to_d has 10 variables added.
2024-04-21 15:09:52 [INFO] (SCIP Solver): Began adding variable transport_d_to_c to model
2024-04-21 15:09:52 [INFO] (SCIP Solver): Added variable transport_d_to_c to model
202

presolving:
(round 1, fast)       5 del vars, 12 del conss, 0 add conss, 140 chg bounds, 0 chg sides, 0 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 2, fast)       5 del vars, 12 del conss, 0 add conss, 140 chg bounds, 0 chg sides, 70 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 3, exhaustive) 5 del vars, 12 del conss, 0 add conss, 140 chg bounds, 0 chg sides, 70 chg coeffs, 70 upgd conss, 0 impls, 0 clqs
   (0.0s) probing cycle finished: starting next cycle
   Deactivated symmetry handling methods, since SCIP was built without symmetry detector (SYM=none).
presolving (4 rounds: 4 fast, 2 medium, 2 exhaustive):
 5 deleted vars, 12 deleted constraints, 0 added constraints, 140 tightened bounds, 0 added holes, 0 changed sides, 70 changed coefficients
 70 implications, 0 cliques
presolved problem has 75 variables (5 bin, 0 int, 0 impl, 70 cont) and 87 constraints
     70 constraints of type <varbound>
     17 constraints of type <linear>
Presolving Time: 0.00

 time | node  | left

2024-04-21 15:09:52 [INFO] (SCIP Solver): Added constraint distribution_cost_incurred_constraint to model
2024-04-21 15:09:52 [INFO] (SCIP Solver): Constraint distribution_cost_incurred_constraint has 5 constraints added.
2024-04-21 15:09:52 [INFO] (SCIP Solver): Began adding constraint distribution_open_constraint to model.
2024-04-21 15:09:52 [INFO] (SCIP Solver): Added constraint distribution_open_constraint to model
2024-04-21 15:09:52 [INFO] (SCIP Solver): Constraint distribution_open_constraint has 5 constraints added.
2024-04-21 15:09:52 [INFO] (SCIP Solver): Began adding constraint distribution_status_by_d_to_c_supply to model.
2024-04-21 15:09:52 [INFO] (SCIP Solver): Added constraint distribution_status_by_d_to_c_supply to model
2024-04-21 15:09:52 [INFO] (SCIP Solver): Constraint distribution_status_by_d_to_c_supply has 60 constraints added.
2024-04-21 15:09:52 [INFO] (SCIP Solver): Began adding constraint distribution_status_constrained_by_m_to_d_supply to model.
2024-04-21

presolving:
(round 1, fast)       5 del vars, 12 del conss, 0 add conss, 140 chg bounds, 0 chg sides, 0 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 2, fast)       6 del vars, 26 del conss, 0 add conss, 140 chg bounds, 0 chg sides, 56 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 3, fast)       7 del vars, 26 del conss, 0 add conss, 140 chg bounds, 0 chg sides, 56 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 4, fast)       7 del vars, 26 del conss, 0 add conss, 140 chg bounds, 1 chg sides, 56 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 5, fast)       13 del vars, 27 del conss, 0 add conss, 140 chg bounds, 1 chg sides, 56 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 6, fast)       20 del vars, 27 del conss, 0 add conss, 140 chg bounds, 6 chg sides, 56 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 7, fast)       20 del vars, 27 del conss, 0 add conss, 140 chg bounds, 13 chg sides, 56 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 8, exhaustive) 20 del vars, 27 de

2024-04-21 15:09:52 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (2, 0, 1) has value 0.0
2024-04-21 15:09:52 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (2, 0, 2) has value 0.0
2024-04-21 15:09:52 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (2, 0, 3) has value 0.0
2024-04-21 15:09:52 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (2, 0, 4) has value 0.0
2024-04-21 15:09:52 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (2, 0, 5) has value 0.0
2024-04-21 15:09:52 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (2, 0, 6) has value 0.0
2024-04-21 15:09:52 [INFO] (SCIP Solver): Variable t

presolving:
(round 1, fast)       5 del vars, 12 del conss, 0 add conss, 140 chg bounds, 0 chg sides, 0 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 2, fast)       6 del vars, 26 del conss, 0 add conss, 140 chg bounds, 0 chg sides, 56 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 3, fast)       7 del vars, 26 del conss, 0 add conss, 140 chg bounds, 0 chg sides, 56 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 4, fast)       7 del vars, 26 del conss, 0 add conss, 140 chg bounds, 1 chg sides, 56 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 5, fast)       13 del vars, 27 del conss, 0 add conss, 140 chg bounds, 1 chg sides, 56 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 6, fast)       20 del vars, 27 del conss, 0 add conss, 140 chg bounds, 6 chg sides, 56 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 7, fast)       20 del vars, 27 del conss, 0 add conss, 140 chg bounds, 13 chg sides, 56 chg coeffs, 0 upgd conss, 0 impls, 0 clqs
(round 8, exhaustive) 20 del vars, 27 de

2024-04-21 15:09:53 [INFO] (SCIP Solver): Variable transport_m_to_d at indices ('s_distribution_sites', 'time_index', 's_manufacturing_sites'): (4, 8, 0) has value -0.0
2024-04-21 15:09:53 [INFO] (SCIP Solver): Variable transport_m_to_d at indices ('s_distribution_sites', 'time_index', 's_manufacturing_sites'): (4, 8, 1) has value 0.0
2024-04-21 15:09:53 [INFO] (SCIP Solver): Variable transport_m_to_d at indices ('s_distribution_sites', 'time_index', 's_manufacturing_sites'): (4, 9, 0) has value 0.0
2024-04-21 15:09:53 [INFO] (SCIP Solver): Variable transport_m_to_d at indices ('s_distribution_sites', 'time_index', 's_manufacturing_sites'): (4, 9, 1) has value 0.0
2024-04-21 15:09:53 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (0, 0, 0) has value -0.0
2024-04-21 15:09:53 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (0, 0, 1) has value -0.0
2024-04-2

presolving:
(round 1, fast)       5 del vars, 30 del conss, 0 add conss, 1400 chg bounds, 0 chg sides, 0 chg coeffs, 0 upgd conss, 0 impls, 90 clqs
(round 2, fast)       5 del vars, 30 del conss, 0 add conss, 1400 chg bounds, 0 chg sides, 700 chg coeffs, 0 upgd conss, 0 impls, 90 clqs
(round 3, exhaustive) 5 del vars, 35 del conss, 0 add conss, 1400 chg bounds, 5 chg sides, 700 chg coeffs, 0 upgd conss, 0 impls, 90 clqs
(round 4, exhaustive) 5 del vars, 35 del conss, 0 add conss, 1400 chg bounds, 5 chg sides, 700 chg coeffs, 785 upgd conss, 0 impls, 90 clqs
(round 5, exhaustive) 5 del vars, 75 del conss, 0 add conss, 1400 chg bounds, 5 chg sides, 700 chg coeffs, 785 upgd conss, 700 impls, 90 clqs
(round 6, medium)     10 del vars, 75 del conss, 0 add conss, 1400 chg bounds, 10 chg sides, 705 chg coeffs, 785 upgd conss, 700 impls, 90 clqs
   (0.0s) probing cycle finished: starting next cycle
   Deactivated symmetry handling methods, since SCIP was built without symmetry detector (SYM=no

2024-04-21 15:09:53 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (2, 1, 10) has value 0.0
2024-04-21 15:09:53 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (2, 1, 11) has value 0.0
2024-04-21 15:09:53 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (2, 2, 0) has value 0.0
2024-04-21 15:09:53 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (2, 2, 1) has value 0.0
2024-04-21 15:09:53 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (2, 2, 2) has value 0.0
2024-04-21 15:09:53 [INFO] (SCIP Solver): Variable transport_d_to_c at indices ('s_distribution_sites', 'time_index', 's_customers'): (2, 2, 3) has value 0.0
2024-04-21 15:09:53 [INFO] (SCIP Solver): Variable

Look Back Solve Total:16964.0, Daily Solve Total:17655.0, Difference: 691.0
Final DC States for Each Decision Point:
Daily Decision Point Dist 0: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Global Decision Point Dist 0: -0.0
Daily Decision Point Dist 1: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Global Decision Point Dist 1: -0.0
Daily Decision Point Dist 2: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Global Decision Point Dist 2: 0.0
Daily Decision Point Dist 3: [-0.0, 0.0, 0.0, -0.0, 0.0, 0.0, -0.0, 0.0, 0.0, -0.0]
Global Decision Point Dist 3: 1.0
Daily Decision Point Dist 4: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Global Decision Point Dist 4: 0.0
Avg dc open on daily simulation: [0.0, 1.0, 0.0, 0.0, 0.0]


In [56]:
# Output average decisions from daily simulations
print('Average Distribution Site Open Decision over 1000 Simulations and 10 Days:')
for d in range(num_distribution_sites):
    print(f'Distribution Site {d}: {average_dc_open[d]:.2f}')


total_sum = sum(average_difference)
num_elements = len(average_difference)
average = total_sum / num_elements if num_elements > 0 else 0


Average Distribution Site Open Decision over 1000 Simulations and 10 Days:
Distribution Site 0: 0.00
Distribution Site 1: 1.00
Distribution Site 2: 0.00
Distribution Site 3: 0.00
Distribution Site 4: 0.00


In [57]:
# Output results from global simulation
print('\nGlobal Simulation Results:')
print(f'Average Difference:{average}')
print('Open Distribution Sites:', global_dc_open)


Global Simulation Results:
Average Difference:691.0
Open Distribution Sites: [-0.0, -0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, -0.0, 0.0, -0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


RL Model

In [58]:
# Define RL parameters
learning_rate = 0.1
discount_factor = 0.9
epsilon = 0.1
num_episodes = 1000

# Initialize Q-table
num_actions = 2  # Open or close distribution site
num_states = num_days * (num_distribution_sites + num_manufacturing_sites + num_customers) * 2  # Number of possible states (open/closed for each distribution site on each day)
q_table = np.zeros((num_states, num_actions))

# Initialize the array for manufacturing site capacity
mfg_site_capacity_array = np.array(mfg_site_capacity)[:, np.newaxis]

# Helper function to convert state tuple to index
def state_to_index(state):
    day, status = state
    distribution_status, manufacturing_status, customer_demand = np.split(status, [num_distribution_sites, num_distribution_sites + num_manufacturing_sites])
    index = day * num_states
    index += np.sum(distribution_status * 2)
    index += np.sum(manufacturing_status * 2) * num_distribution_sites
    index += np.sum(customer_demand) * num_distribution_sites * num_manufacturing_sites
    return index

# RL training
for episode in range(num_episodes):
    total_cost = 0

    # Initialize state
    state = (0, np.zeros(num_states, dtype=int))

    for day in range(num_days):
        # Choose action (epsilon-greedy policy)
        if np.random.rand() < epsilon:
            action = np.random.randint(num_actions)
        else:
            index = state_to_index(state)
            action = np.argmax(q_table[index])

        # Update state
        next_state = (day + 1, state[1].copy())
        next_state[1][action] = 1 - next_state[1][action]

        total_dc_cost = 0
        for distribution, dc_solution in enumerate(next_state[1][:num_distribution_sites]):
            if dc_solution == 1:
                if days_dc_open[distribution] == 0:
                    total_dc_cost += distribution_opening_costs[distribution]
                days_dc_open[distribution] += 1
                if days_dc_open[distribution] >= decision_rolling_period:
                    days_dc_open[distribution] = 0
        total_cost += total_dc_cost

        # Calculate transportation costs from manufacturing sites to distribution sites based on their status
        manufacturing_to_distribution_costs = np.sum(transport_cost_m_to_d * mfg_site_capacity_array * next_state[1][num_distribution_sites:num_distribution_sites+num_manufacturing_sites], axis=0)

        # Calculate transportation costs from distribution sites to customers based on their demand and status
        distribution_to_customer_costs = np.sum(transport_cost_d_to_c * np.dot(next_state[1][:num_distribution_sites][:, np.newaxis], mean_demand[np.newaxis, :]), axis=(0, 1))

        # Add transportation costs to the total cost
        total_cost += np.sum(manufacturing_to_distribution_costs) + distribution_to_customer_costs

        # Update Q-table
        index = state_to_index(state)
        next_index = state_to_index(next_state)
        best_next_action = np.argmax(q_table[next_index])
        q_table[index][action] += learning_rate * (total_cost + discount_factor * q_table[next_index][best_next_action] - q_table[index][action])

        # Move to the next state
        state = next_state

print("Training complete.")

print("Training complete.")

# Evaluate RL agent
# Assuming the RL agent will choose actions greedily
total_costs = []
for episode in range(num_episodes):
    state = (0, np.zeros(num_states, dtype=int))
    total_cost = 0
    for day in range(num_days):
        index = state_to_index(state)
        action = np.argmax(q_table[index])
        total_dc_cost = 0
        for distribution, dc_solution in enumerate(state[1]):
            if dc_solution == 1:
                if days_dc_open[distribution] == 0:
                    total_dc_cost += distribution_opening_costs[distribution]
                days_dc_open[distribution]


ValueError: operands could not be broadcast together with shapes (2,5) (2,) 